In [118]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [119]:
import pandas as pd

df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [120]:
#data Analysis
df.groupby("Sex")["Survived"].mean()
df.groupby("Pclass")["Survived"].mean()

df.groupby(["Pclass", "Sex"])["Survived"].mean()
df.groupby("Survived")["Age"].mean()
df.groupby("Embarked")["Survived"].mean()
df.groupby("Survived")["Fare"].mean()
df.isnull().sum()
df["Embarked"].mode()


0    S
Name: Embarked, dtype: object

In [121]:
#data preparation

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop("Cabin", axis=1)
df.isnull().sum()
df["Sex"].unique()



array(['male', 'female'], dtype=object)

In [122]:
# convert caterical data into numbers
df["Sex"] = df["Sex"].map({"female": 0, "male": 1})
df["Sex"].head()
df = pd.get_dummies(df, columns=["Embarked"], dtype=int)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,0,0,1
4,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,0,0,1


In [123]:
# Splitting input and output data
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked_C",
    "Embarked_Q",
    "Embarked_S"
]

X = df[features]
y = df["Survived"]
X.head()
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Pclass      891 non-null    int64  
 1   Sex         891 non-null    int64  
 2   Age         891 non-null    float64
 3   SibSp       891 non-null    int64  
 4   Parch       891 non-null    int64  
 5   Fare        891 non-null    float64
 6   Embarked_C  891 non-null    int64  
 7   Embarked_Q  891 non-null    int64  
 8   Embarked_S  891 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 62.8 KB


In [124]:
#train and test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape)
print(X_test.shape)

(712, 9)
(179, 9)


In [125]:
#model training
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)
print(model.coef_)
model.intercept_

[[-9.35458835e-01 -2.59231752e+00 -3.05172545e-02 -2.94206927e-01
  -1.09465182e-01  2.54926275e-03  1.89125769e-01  3.89592108e-02
  -2.32339805e-01]]


array([4.35901491])

In [126]:
# prediction
y_pred = model.predict(X_test)

print(y_pred[:10])

[0 0 0 1 1 1 1 0 1 1]


In [127]:
# check accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8100558659217877


In [128]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[90 15]
 [19 55]]


In [129]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Precision: 0.7857142857142857
Recall: 0.7432432432432432
F1: 0.7638888888888888


In [132]:
test_df = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
# =========================

# =========================
# 2. Handle missing values
# =========================

test_df["Age"] = test_df["Age"].fillna(test_df["Age"].median())

test_df["Fare"] = test_df["Fare"].fillna(test_df["Fare"].median())

test_df["Embarked"] = test_df["Embarked"].fillna("S")


# =========================
# 3. Encode categorical data
# =========================

test_df["Sex"] = test_df["Sex"].map({
    "female": 0,
    "male": 1
})

test_df = pd.get_dummies(
    test_df,
    columns=["Embarked"],
    dtype=int
)


# =========================
# 4. Drop unnecessary column
# =========================

test_df = test_df.drop("Cabin", axis=1)


# =========================
# 5. Select same features
# =========================

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked_C",
    "Embarked_Q",
    "Embarked_S"
]

X_test_kaggle = test_df[features]


# =========================
# 6. Check for missing values
# =========================

print("Missing values:")
print(X_test_kaggle.isnull().sum())


# =========================
# 7. Make predictions
# =========================

predictions = model.predict(X_test_kaggle)


# =========================
# 8. Create submission
# =========================

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)


# =========================
# 9. Check submission
# =========================

print("\nSubmission preview:")
print(submission.head())

print("\nSubmission shape:")
print(submission.shape)

Missing values:
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_C    0
Embarked_Q    0
Embarked_S    0
dtype: int64

Submission preview:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1

Submission shape:
(418, 2)
